# 00 — Build the RAG dataset (retrieval corpus)

Creates **`data/processed/rag_corpus.csv`** — the pool of labelled *precedent* loans the RAG notebooks retrieve from. It is the **full large dataset with every evaluation batch removed**, so no eval (or test) loan can ever be retrieved.

**Source pool (auto):**
1. The full 2012–2014 LendingClub frame from the raw `data/raw/accepted_2007_to_2018Q4.csv.gz` (via `sample_generation._build_frame`). The real, large corpus.
2. **Dev fallback** — if the raw file is absent: the committed `tuning_sample` (~100 rows, same 35-col schema, disjoint from the `robustness_batch` we evaluate on). Lets everything run before the raw file is added.

The **`robustness_batch`** (the evaluation set) is always removed so it can never be retrieved as a precedent; the held-out **`test_batch`** is removed too (read only to keep it out of the corpus — it is reserved for Phase 4 and never evaluated in Phase 5). Zero overlap with the eval set is asserted.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import rag_utils as R

In [ ]:
# Build (force=True regenerates). Drop the raw .csv.gz into data/raw/ and rerun
# with force=True to swap the dev fallback for the full corpus.
corpus = R.build_rag_corpus(force=True)

# Leakage guard: the corpus must be disjoint from the evaluation set (robustness_batch).
# The held-out test_batch is excluded by build_rag_corpus too, but is never evaluated
# in Phase 5 (strict-holdout protocol — Phase 4 only).
robustness = pd.read_csv(R.ROBUSTNESS_PATH)
R.assert_no_leakage(corpus, robustness)
print(f'RAG corpus rows       : {len(corpus)}')
print(f'Robustness (eval) rows: {len(robustness)}')
print('Leakage check         : PASSED (corpus ∩ robustness = ∅)')

In [ ]:
# Quick profile of the corpus
co = int((corpus['loan_status'] == 0).sum())
print(f'Charged Off: {co}  |  Fully Paid: {len(corpus) - co}')
corpus.head()